# Exploring Library Catalogues as Data
## Part I. Preparation
### Install and import the necessary libraries

[%pip](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-pip) is a "magic command" in Jupyter, that runs the [pip](https://pypi.org/project/pip/) package installer inside the notebook.

In [7]:
%pip install pymarc pandas lxml matplotlib matplotlib-venn numpy

Note: you may need to restart the kernel to use updated packages.


As usual in Python we should start with importing the Python libraries we would like to utilize in the script
* `urllib.request` is a library for opening URLs, [https://docs.python.org/3/library/urllib.request.html](https://docs.python.org/3/library/urllib.request.html) 
* `os` contains miscellaneous operating system interfaces, [https://docs.python.org/3/library/os.html](https://docs.python.org/3/library/os.html)
* `gzip` supports operations on gzip files, [https://docs.python.org/3/library/gzip.html](https://docs.python.org/3/library/gzip.html)
* `shutil` provides high-level file operations, [https://docs.python.org/3/library/shutil.html](https://docs.python.org/3/library/shutil.html)
* `re` provides regular expression operations, [https://docs.python.org/3/library/re.html](https://docs.python.org/3/library/re.html)

In [8]:
import urllib.request
import os
import gzip
import shutil
import re

### Download a single file

We should specify the URL of the file we would like to download:

In [9]:
url = 'https://metadata.library.yale.edu/MARCXML/bib_20250706_full/bib_20250706_full_000_00.xml.gz'

In our machine, it will be located in a specific directory (we call it `target_dir`), and if it is not yet existing, we should create it.

In [10]:
target_dir = 'raw-data/yale'
if not os.path.exists(target_dir):
    os.makedirs(target_dir)

Then we should specify the file in our local machine. We extract it from the URL with a regular expression. `/([^/]+)$` means find a slash character (`/`) followed by one or more non-slash characters (`[^/]+`) till the end of the string (`$`), and put these characters into a group `(...)`. With this we specify the file name. With `group(1)` we can extract the content of the first (and in this case the only) group. Finally, we concatenate the directory and file names with an [f-string](https://realpython.com/python-f-strings/).

In [11]:
file_name = re.search('/([^/]+)$', url).group(1)
target_file = f'{target_dir}/{file_name}'

The act of downloading is pretty simple, it saves the content of the URL into the specified file:

In [12]:
urllib.request.urlretrieve(url, target_file)

('raw-data/yale/bib_20250706_full_000_00.xml.gz',
 <http.client.HTTPMessage at 0x7f613152b0e0>)

As we would like to work with an XML file and not a compressed file (which would be also possible, but not discussed in this lesson), we should extract it. It needs some steps. With `gzip.open()` we open the archive file in binary read mode (it behaves similarly to other file read operations in Python), and we specify a file handle (`f_in`). We should also specify the name of the uncompressed file with the help of another regular expression. `re.sub()` substitutes strings. Here we are looking for the `.gz` extension in the file name, and replace it with an empty string - in other words, we remove it. Note: in regular expression `.` (dot character) has a special meaning: it fits any character. If we want to mean the real dot, we should escape this interpretation with the backslashes. We put an `r` prefix before the search string. This refers to the so-called _r-string_ or [raw string notation](https://mimo.org/glossary/python/raw-strings) that treats backslashes (`\`) as literal characters rather than escape sequences, otherwise we should add double backslashes, to behave as escape sequence in regular expressions. Finally, we open a binary file for writing and utilize the `shutil.copyfileobj()` method to copy the content. 

In [13]:
with gzip.open(target_file, 'rb') as f_in:
    uncompressed_file = re.sub(r'\.gz$', '', target_file)
    with open(uncompressed_file, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

Our final step is to remove the unwanted compressed file:

In [14]:
os.remove(target_file)

### Download multiple files

Import additional libraries
* `sys` contains system-specific parameters and functions, [https://docs.python.org/3/library/sys.html](https://docs.python.org/3/library/sys.html)
* `lxml` responsible for handling XML and HTML, [https://lxml.de/import lxml.html](https://lxml.de/import lxml.html)

In [15]:
import sys
import lxml.html

We create a configuration with default values:

In [16]:
configuration = {
  'index_page': 'https://metadata.library.yale.edu/MARCXML/bib_20250706_full',
  'target_dir': 'raw-data/yale'
}

Because we will download multiple files, it would be useful to separate the code into a function that accepts a file name, and utilizes the configuration object

In [17]:
def download_file(file_name):
    # We start with the function’s documentation:
    """
    Downloads a file, saves it into a directory, uncompresses it and deletes the compressed version.
    The base URL and the target directory come from the configuration object.
    Parameters                              
    ----------
    file_name : str
        the name of the downloadable file
    """
    # Next we set the variables based on the input parameter and the configuration. 
    # A log entry will inform the user about the process:
    remote_file = configuration['index_page'] + '/' + file_name
    local_file = configuration['target_dir'] + '/' + file_name
    uncompressed_file = re.sub(r'\.gz', '', local_file)
    print(f'downloading {remote_file} to {uncompressed_file} ...')

    # The bulk of the function repeats what we saw in the single file download,
    # with a check (launch download if neither the gzip nor the xml file are available)
    # and a try-except block. This later catches network problems and informs the user.
    # If we would not put the functionality inside that block an error would stop the script itself.
    if not os.path.exists(local_file) and not os.path.exists(uncompressed_file):
        try:
            urllib.request.urlretrieve(remote_file, local_file)

            with gzip.open(local_file, 'rb') as f_in:
                with open(uncompressed_file, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)

        except urllib.error.HTTPError as e:
            print("A network problem occured: ", e)
    
    if os.path.exists(local_file):
        os.remove(local_file)

And finally we should fetch the index page, extract links to the .gz files, and call the `download_file()` function. This time we do not save the result of URL request, but save it into memory as a [HTTPResponse](https://docs.python.org/3/library/http.client.html#http.client.HTTPResponse) object. We read its content into a string, then the `lxml` library parses the HTML structure allowing us to run searches with an XPath expression. `body/table/tr/td/a` finds all links inside the page tables. We iterate over them, extracting the `href` attribute of each link, and if they end with `.gz`, calling the download function.

For the purpose of this tutorial we download only the first two files, so we introduce a `counter` variable that keeps the number of successfully downloaded files.

In [18]:
def main():
    if not os.path.exists(configuration['target_dir']):
        os.makedirs(configuration['target_dir'])

    with urllib.request.urlopen(configuration['index_page']) as response:
        content = response.read()
        doc = lxml.html.fromstring(content)
        items = doc.findall('body/table/tr/td/a', {})
        counter = 0
        for item in items:
            if counter < 2:
                file_name = item.get('href')
                if re.search(r'\.gz$', file_name):
                    download_file(file_name)
                    counter += 1

In [19]:
main()

downloading https://metadata.library.yale.edu/MARCXML/bib_20250706_full/bib_20250706_full_000_00.xml.gz to raw-data/yale/bib_20250706_full_000_00.xml ...
downloading https://metadata.library.yale.edu/MARCXML/bib_20250706_full/bib_20250706_full_000_01.xml.gz to raw-data/yale/bib_20250706_full_000_01.xml ...


### Function 1: extract_to_dataframe
We use two more libraries:
* [PyMARC](https://gitlab.com/pymarc/pymarc) reads and parses MARC records. This lesson works mainly with the MARCXML form.
* [pandas](https://pandas.pydata.org/) provides the DataFrame, the tabular structure we reshape records into.

In [20]:
from pymarc import map_xml
import pandas as pd

Wrapping the whole extraction in a function lets us reuse it: each call takes a file path and returns a fresh DataFrame, so we can run it on as many files as we like. This function also adds subject extraction to the fields we pulled earlier.

In [21]:
def extract_to_dataframe(*file_paths):
    """Read one or more MARCXML files. Return a DataFrame with one row
    per record and columns: id, title, author, subjects (pipe-separated)."""
    rows = []

    def process_record(record):
        field_100 = record.get('100')
        if field_100 is not None and field_100.get('a') is not None:
            author = field_100.get('a')
        else:
            author = None

        subject_values = [s.get('a') for s in record.subjects if s.get('a') is not None]

        rows.append({
            'id': record.get('001').value(),
            'title': record.title,
            'author': author,
            'subjects': '|'.join(subject_values),
        })

    for path in file_paths:
        map_xml(process_record, path)

    return pd.DataFrame(rows)

Extract data frames from two downloaded MARCXML files:

In [22]:
df_a = extract_to_dataframe('raw-data/yale/bib_20250706_full_000_00.xml')
df_b = extract_to_dataframe('raw-data/yale/bib_20250706_full_000_01.xml')

We can check what we have in the data frame. pandas have some useful properties and functions:
* `shape` returns the number of rows and number of columns
* `head()` returns the first 5 rows
* `tail()` returns the last 5 rows

Note: when we print out the output of the two functions the content of the cells will be truncated - but it is only for the print, the data is not truncated!

In [24]:
print(df_a.shape)
df_a.head()

(200000, 4)


,id,title,author,subjects
0,2,Die Streitkräfte der NATO auf dem Territorium...,NaN,North Atlantic Treaty Organization.|North Atla...
1,3,Arqueoecología : el hombre en los ecosistemas...,"D'Antoni, Héctor L.",Paleoecology.
2,4,An economic history of Spain /,"Vicens Vives, Jaime,",Spain|Economic history.|Spain.
3,5,Appleton's cyclopaedia of American biography.,NaN,America|United States
4,6,"The architectural planning of St. Petersburg,","Egorov, I︠U︡. A.",City planning


In [25]:
df_a.tail()

,id,title,author,subjects
199995,202830,Slawische Personennamen in mittelalterlichen Q...,"Schlimpert, Gerhard.","Slavic languages|Names, Personal|Names, Personal"
199996,202831,Science and the sociology of knowledge /,"Mulkay, M. J.",Science
199997,202832,Stoned images /,"Hoefer, Hans.","Sculpture|Decoration and ornament, Architectural"
199998,202833,Décimas /,"Marrero Cabello, Pablo.",
199999,202834,"Bibliography on the fatigue of materials, comp...","Mann, J. Y.",Materials


Saving files the data frames to CSV. `to_csv()` saves the content of a data frame into a CSV file. `index = False` prevents writing the row names (the data frame index) into the file. Unfortunately, the default value of this argument is `True`, which makes CSV a bit weird, so we have to set `False`. 

In [26]:
df_a.to_csv('bib_20250706_full_000_00.csv', index = False)
df_b.to_csv('bib_20250706_full_000_01.csv', index = False)

To read a files into a data frame pandas provides the `read_csv()` function. We should pass at least the file name, and assign the result into a variable. We print out the first five rows to be sure that the new data frame is the same as what we had after extraction.

In [27]:
df = pd.read_csv('bib_20250706_full_000_00.csv')
print(df.shape)
df.head()

(200000, 4)


,id,title,author,subjects
0,2,Die Streitkräfte der NATO auf dem Territorium...,NaN,North Atlantic Treaty Organization.|North Atla...
1,3,Arqueoecología : el hombre en los ecosistemas...,"D'Antoni, Héctor L.",Paleoecology.
2,4,An economic history of Spain /,"Vicens Vives, Jaime,",Spain|Economic history.|Spain.
3,5,Appleton's cyclopaedia of American biography.,NaN,America|United States
4,6,"The architectural planning of St. Petersburg,","Egorov, I︠U︡. A.",City planning


In [28]:
df.tail()

,id,title,author,subjects
199995,202830,Slawische Personennamen in mittelalterlichen Q...,"Schlimpert, Gerhard.","Slavic languages|Names, Personal|Names, Personal"
199996,202831,Science and the sociology of knowledge /,"Mulkay, M. J.",Science
199997,202832,Stoned images /,"Hoefer, Hans.","Sculpture|Decoration and ornament, Architectural"
199998,202833,Décimas /,"Marrero Cabello, Pablo.",NaN
199999,202834,"Bibliography on the fatigue of materials, comp...","Mann, J. Y.",Materials
